In [116]:
from pathlib import Path
import pandas as pd
from rdkit import Chem
from rdkit.Chem import PandasTools, Descriptors, rdchem, rdMolDescriptors
from rdkit.ML.Descriptors import MoleculeDescriptors
import numpy as np
from sklearn.model_selection import cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LassoCV

def mol_to_feat(mol):
    #找我认为的重要键
    NO_single = NO_double = 0
    NN_single = NN_double = 0
    sum = 0
    for i in mol.GetBonds():
        pre = i.GetBeginAtom().GetSymbol()
        nxt = i.GetEndAtom().GetSymbol()
        link = i.GetBondType()
        if pre == "O":
            pre, nxt = nxt, pre
        if pre == "N" and nxt == "O":
            if link == rdchem.BondType.SINGLE:
                NO_single += 1
            else:
                NO_double += 1
        elif pre == "N" and nxt == "N":
            if link == rdchem.BondType.SINGLE:
                NN_single += 1
            else:
                NN_double += 1
        sum += 1

    #算各种原子的个数
    mqn = rdMolDescriptors.MQNs_(mol)
    N = mqn[7] + mqn[8]
    O = mqn[9] + mqn[10]
    C = mqn[0]
    mol = Chem.AddHs(mol)
    H = mol.GetNumAtoms() - mol.GetNumHeavyAtoms()
    
    feat = [
        (C * 2 + H / 2 - O) * 16 / Descriptors.MolWt(mol),
        N * 14 / Descriptors.MolWt(mol),
        NO_single / sum, NO_double / sum,
        NN_single / sum, NN_double / sum
    ]
    return feat

file = Path.cwd().parent / "data" / "test.xlsx"
df = pd.read_excel(file)
PandasTools.AddMoleculeColumnToFrame(df, "SMILES", "ROMol", False)
x_all = np.stack(df["ROMol"].apply(mol_to_feat).tolist(), axis = 0)
y_all = np.array(df["Q(cal/g)"], dtype = np.float64)

for i in range(5):
    pipe = make_pipeline(PolynomialFeatures(i + 1), StandardScaler(), LassoCV(cv = 5))
    scores = cross_validate(pipe, x_all, y_all, cv = 10, scoring = ["neg_mean_squared_error", "r2"], n_jobs = -1)
    mse = np.abs(scores["test_neg_mean_squared_error"]).mean()
    r2 = scores["test_r2"]
    print(i + 1, mse, r2.mean(), r2.std())
    print(r2)


1 66942.96184910256 0.5424084342513057 0.2503883559403863
[0.5618413  0.69475966 0.38289103 0.85226222 0.03398026 0.34826668
 0.83265966 0.66578862 0.31882664 0.73280827]
2 54107.005851261216 0.6302357466581417 0.14433907382637257
[0.61508349 0.8292545  0.66080496 0.69377027 0.24021257 0.62460677
 0.71995534 0.59601299 0.65073445 0.67192213]
3 38881.19862583897 0.7806983361457942 0.1417908999940754
[0.4154607  0.88517822 0.84261038 0.76390382 0.67810865 0.92573624
 0.79062519 0.82963236 0.76031606 0.91541175]
4 35755.44242794518 0.7959127418760804 0.12488236612481968
[0.47747319 0.86836794 0.84485072 0.83944246 0.69583471 0.9092425
 0.84251485 0.84681623 0.72652777 0.90805704]
5 33161.25722538953 0.7879084754603213 0.13341970299161454
[0.61789646 0.89401459 0.7767481  0.87850816 0.47425587 0.85638261
 0.90518228 0.87019232 0.75065661 0.85524775]


In [2]:
from pathlib import Path
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors, Descriptors, rdchem
import numpy as np

def mol_to_feat(mol):
    NO_single = NO_double = 0
    NN_single = NN_double = 0
    sum = 0
    for i in mol.GetBonds():
        pre = i.GetBeginAtom().GetSymbol()
        nxt = i.GetEndAtom().GetSymbol()
        link = i.GetBondType()
        if pre == "O":
            pre, nxt = nxt, pre
        if pre == "N" and nxt == "O":
            if link == rdchem.BondType.SINGLE:
                NO_single += 1
            else:
                NO_double += 1
        elif pre == "N" and nxt == "N":
            if link == rdchem.BondType.SINGLE:
                NN_single += 1
            else:
                NN_double += 1
        sum += 1

    mqn = rdMolDescriptors.MQNs_(mol)
    N = mqn[7] + mqn[8]
    O = mqn[9] + mqn[10]
    C = mqn[0]
    mol = Chem.AddHs(mol)
    H = mol.GetNumAtoms() - mol.GetNumHeavyAtoms()

    feat = [
        (C * 2 + H / 2 - O) * 16 / Descriptors.MolWt(mol),
        N * 14 / Descriptors.MolWt(mol),
        NO_single / sum, NO_double / sum,
        NN_single / sum, NN_double / sum
    ]
    return feat
    

f = mol_to_feat(Chem.MolFromSmiles("O=[N+]([O-])Oc1nn2cnnc2n1O[N+](=O)[O-]"))
f

[0.03461944574267366,
 0.42408821034775235,
 0.29411764705882354,
 0.11764705882352941,
 0.0,
 0.11764705882352941]

In [ ]:
from pathlib import Path
import pandas as pd
from rdkit import Chem
from rdkit.ML.Descriptors import MoleculeDescriptors
from rdkit.Chem import PandasTools, Descriptors, rdchem, rdMolDescriptors
from sklearn.feature_selection import SelectKBest, mutual_info_regression
from sklearn.ensemble import RandomForestRegressor
import numpy as np

def mol_to_feat(mol):
    #找我认为的重要键
    NO_single = NO_double = 0
    NN_single = NN_double = 0
    sum = 0
    for i in mol.GetBonds():
        pre = i.GetBeginAtom().GetSymbol()
        nxt = i.GetEndAtom().GetSymbol()
        link = i.GetBondType()
        if pre == "O":
            pre, nxt = nxt, pre
        if pre == "N" and nxt == "O":
            if link == rdchem.BondType.SINGLE:
                NO_single += 1
            else:
                NO_double += 1
        elif pre == "N" and nxt == "N":
            if link == rdchem.BondType.SINGLE:
                NN_single += 1
            else:
                NN_double += 1
        sum += 1

    #算各种原子的个数
    mqn = rdMolDescriptors.MQNs_(mol)
    N = mqn[7] + mqn[8]
    O = mqn[9] + mqn[10]
    C = mqn[0]
    mol = Chem.AddHs(mol)
    H = mol.GetNumAtoms() - mol.GetNumHeavyAtoms()

    #算重要基团
    nitrate = len(mol.GetSubstructMatches(Chem.MolFromSmarts("[OX2][N+](=O)[O-]")))
    nitro = len(mol.GetSubstructMatches(Chem.MolFromSmarts("[N+](=O)[O-]"))) - nitrate
    nitroso = len(mol.GetSubstructMatches(Chem.MolFromSmarts("[N;R0]=[O;D1]")))
    azide = len(mol.GetSubstructMatches(Chem.MolFromSmarts("[N-]=[N+]=N")))

    Wt = Descriptors.MolWt(mol)
    HeavyAtoms = Descriptors.HeavyAtomCount(mol)

    feat = [
        (C * 2 + H / 2 - O) * 16 / Wt,
        Descriptors.fr_Ar_N(mol) * 14 / Wt,
        nitrate / HeavyAtoms, nitro / HeavyAtoms, nitroso / HeavyAtoms, azide / HeavyAtoms,
        NN_single / sum, NN_double / sum, NO_single, NO_double
    ]

    desc_names = ['EState_VSA8', 'VSA_EState3', 'VSA_EState4', 'PEOE_VSA1', 'PEOE_VSA7', 'SlogP_VSA1', 'SlogP_VSA4']
    feat = feat + list(MoleculeDescriptors.MolecularDescriptorCalculator(desc_names).CalcDescriptors(mol))
    
    return feat

file = Path.cwd().parent / "data" / "test.xlsx"
df = pd.read_excel(file)
PandasTools.AddMoleculeColumnToFrame(df, "SMILES", "ROMol", False)

data = df["ROMol"].apply(mol_to_feat).tolist()
desc_name = ["OB*", "fr_Ar_N", "nitrate", "nitro", "nitroso", "azide", "NN_single", "NN_double", "NO_single", "NO_double", 'EState_VSA8', 'VSA_EState3', 'VSA_EState4', 'PEOE_VSA1', 'PEOE_VSA7', 'SlogP_VSA1', 'SlogP_VSA4']
df2 = pd.DataFrame(data, columns = desc_name)

corr = df2.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k = 1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.85)]
df2 = df2.drop(columns = to_drop)

Importances = RandomForestRegressor().fit(df2.values, df["Q(cal/g)"]).feature_importances_
feature_importances = pd.Series(Importances, index = df2.columns)
print(feature_importances.sort_values(ascending = False))

kbest_selector = SelectKBest(score_func = mutual_info_regression, k = 7)
df2_kbest = kbest_selector.fit_transform(df2.values, df["Q(cal/g)"])
selected_columns = df2.columns[kbest_selector.get_support()]
df3 = pd.DataFrame(df2_kbest, columns = selected_columns)
print(kbest_selector.scores_[kbest_selector.get_support()])
df3

VSA_EState3    0.446487
OB*            0.246361
EState_VSA8    0.093566
nitro          0.036656
NO_double      0.034964
fr_Ar_N        0.032388
PEOE_VSA7      0.024042
NN_double      0.017289
VSA_EState4    0.013089
NN_single      0.012931
PEOE_VSA1      0.012927
SlogP_VSA1     0.011339
NO_single      0.009984
nitrate        0.007977
azide          0.000000
dtype: float64
[0.5941581  0.40034469 0.28237795 0.37008422 0.45389507 0.17135477
 0.15100215]


,OB*,nitro,NO_single,NO_double,VSA_EState3,PEOE_VSA7,SlogP_VSA1
0,1.013243,0.000000,0.0,0.0,0.000000,0.000000,22.797447
1,0.412926,0.066667,1.0,1.0,22.385921,0.000000,11.138533
2,0.374792,0.111111,2.0,2.0,32.128086,15.725183,10.946635
3,1.099989,0.000000,0.0,0.0,22.362116,0.000000,11.029530
4,0.873351,0.117647,2.0,2.0,28.437707,6.042419,0.000000
...,...,...,...,...,...,...,...
83,0.128294,0.115385,4.0,5.0,34.801790,10.575479,10.256405
84,0.682985,0.071429,1.0,1.0,15.317528,0.000000,15.875396
85,0.603383,0.000000,2.0,1.0,5.451154,0.000000,37.578597
86,-0.068350,0.187500,3.0,3.0,42.940208,10.446506,5.719717
